In [4]:
import tensorflow as tf 
import numpy as np 
from tensorflow.keras.layers import TextVectorization 
from tensorflow.keras.utils import get_file 
from tensorflow.keras.layers import Embedding, MultiHeadAttention, Dense, LayerNormalization, Dropout
from tensorflow.keras.models import Model

In [3]:

# 1. Download the text directly from Google's cloud (only ~1MB)
url = 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
path_to_file = tf.keras.utils.get_file('shakespeare.txt', url)

# 2. Read the file into a Python string
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')

# 3. Prove it worked! Let's look at the data
print(f"Total characters in dataset: {len(text)}\n")
print("--- HERE ARE THE FIRST 250 CHARACTERS ---")
print(text[:250])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Total characters in dataset: 1115394

--- HERE ARE THE FIRST 250 CHARACTERS ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



In [13]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.attention = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation='relu'),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.drop1 = Dropout(rate)
        self.drop2 = Dropout(rate)

    def call(self, inputs, training=False):
        attentionOut = self.attention(inputs, inputs, use_causal_mask = True)
        attentionOut = self.drop1(attentionOut, training=training)
        out1 = self.layernorm1(attentionOut + inputs)
        ff_out = self.ffn(out1)
        ff_outn = self.drop2(ff_out, training=training)
        return self.layernorm2(ff_out + out1)

class TransformerModel(Model):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, seq_length):
        super(TransformerModel, self).__init__()
        self.embeddings = Embedding(vocab_size, embed_dim)
        self.pos_encoding = self.positional_encoding(seq_length, embed_dim)
        self.transformerBlocks = [TransformerBlock(embed_dim,num_heads,ff_dim) for _ in range(num_layers)]
        self.dense = Dense(vocab_size)

    def positional_encoding(self, seq_length, embed_dim):
        angle_rads = self.get_angles(np.arange(seq_length)[:, np.newaxis], np.arange(embed_dim)[np.newaxis, :], embed_dim)
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        pos_encoding = angle_rads[np.newaxis, ...]
        return tf.cast(pos_encoding, dtype=tf.float32)

    def get_angles(self, pos, i, embed_dim):
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(embed_dim))
        return pos * angle_rates

    def call(self, inputs, training=False):
        seq_len = tf.shape(inputs)[1]
        x = self.embeddings(inputs)
        x += self.pos_encoding[:, :seq_len, :]
        for transformerblock in self.transformerBlocks:
            x = transformerblock(x, training=training)
        output = self.dense(x)
        return output

In [10]:
vocab_size=10000
seq_length=15

vectorizer = TextVectorization(max_tokens=vocab_size, output_mode='int')
text_ds = tf.data.Dataset.from_tensor_slices([text]).batch(1)
vectorizer.adapt(text_ds)

vectorized_text = vectorizer([text])[0]

print("Vectorized text shape:", vectorized_text.shape) 
print("First 10 vectorized tokens:", vectorized_text.numpy()[:10]) 

Vectorized text shape: (202646,)
First 10 vectorized tokens: [ 89 270 138  36 982 144 673 125  16 106]


2026-06-09 11:12:46.644239: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [11]:
def create_sequences(text, seq_len):
    input_seqs = []
    target_seqs = []

    for i in range(len(text) - seq_len):
        input_seq = text[i:i+seq_len]
        target_seq = text[i+1:i+seq_len+1]
        input_seqs.append(input_seq)
        target_seqs.append(target_seq)
    return np.array(input_seqs), np.array(target_seqs)

X, Y = create_sequences(vectorized_text.numpy(), seq_length)

X = tf.convert_to_tensor(X)
Y = tf.convert_to_tensor(Y)

print("Shape of X: ",X.shape)
print("Shape of Y: ",Y.shape)

Shape of X:  (202631, 15)
Shape of Y:  (202631, 15)


In [14]:
embed_dims=256
num_heads=4
ff_dim = 512
num_layers = 4

model = TransformerModel(vocab_size=vocab_size,embed_dim=embed_dims,num_heads=num_heads,ff_dim=ff_dim,num_layers=num_layers,seq_length=seq_length)

_ = model(tf.random.uniform((1,seq_length),maxval=vocab_size, dtype=tf.int32))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()


Model: "transformer_model_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (1, 15, 256)           │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ ?                      │     1,315,840 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ ?                      │     1,315,840 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_6             │ ?                      │     1,315,840 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ ?                      │     1,315,840 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (1, 15, 10000)         │     2,570,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,393,360 (39.65 MB)

 Trainable params: 10,393,360 (39.65 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
X = X[:10000]
Y = Y[:10000]

In [ ]:
# Early stopping callback to stop training if the loss doesn't improve
from tensorflow.keras.callbacks import EarlyStopping


# Train the transformer model on the full input and target sequences
history = model.fit(X, Y, epochs=10, batch_size=32)

Epoch 1/10
 30/313 ━━━━━━━━━━━━━━━━━━━━ 8:29 2s/step - loss: 14.1172

In [18]:
def generate_text(model, start_string, num_generate=100, temperature=1.0):
    # Convert the start string to a vectorized format
    input_eval = vectorizer([start_string]).numpy()
    
    # Ensure the input length is the same as the model's expected input shape
    if input_eval.shape[1] < seq_length:
        # Pad the input if it's shorter than the expected sequence length
        padding = np.zeros((1, seq_length - input_eval.shape[1]))
        input_eval = np.concatenate((padding, input_eval), axis=1)
    elif input_eval.shape[1] > seq_length:
        # Truncate the input if it's longer than the expected sequence length
        input_eval = input_eval[:, -seq_length:]

    input_eval = tf.convert_to_tensor(input_eval)
    
    # Initialize an empty list to store generated text
    text_generated = []

    # Start generating text
    for i in range(num_generate):
        # Make predictions using the model
        predictions = model(input_eval)

        # Remove only the batch dimension, keep the logits as 2D (batch_size, vocab_size)
        predictions = predictions[0]  # This should be of shape [vocab_size]

        # Apply temperature to predictions
        predictions = predictions / temperature
        
        # Use a categorical distribution to predict the next word
        predicted_id = tf.random.categorical(predictions, num_samples=1)[0, 0].numpy()

        # Update the input tensor to include the predicted word, maintaining the sequence length
        input_eval = np.append(input_eval.numpy(), [[predicted_id]], axis=1)  # Append predicted token
        input_eval = input_eval[:, -seq_length:]  # Keep only the last `seq_length` tokens
        input_eval = tf.convert_to_tensor(input_eval)  # Convert back to tensor

        # Append the predicted word to the generated text
        text_generated.append(vectorizer.get_vocabulary()[predicted_id])

    # Return the generated text starting from the initial seed
    return start_string + ' ' + ' '.join(text_generated)

# Generate text with temperature control
start_string = "To be, or not to be"
generated_text = generate_text(model, start_string, temperature=0.7)  # Lower temperature for more focused predictions
print(generated_text)

To be, or not to be servile damask sooner thinkst needer blue seldshown visit gremio peculiar stoics hereditary swarming my venge unwillingly solace alack breaks overmuch drown behind impatient shines pompous him rapt whereinlet prejudicial catch bats unwedgeable newconceived welshmen rumourd weandi isle my we belly appeal dearest holp camillo them perfidioushe londons swounded unsphere strongframed kernel words folly plaints psalms nit sleight argument leaden trusts nest we threeinch perfumed hates safer certainty ladyshe vulgarly interchangeably insulting compact gaze proserpina marriagedowry spare shines ornaments layt escaped respected insinuate prosperitys oats park welladay linens pointing shal ware fifth but release troubles lookd rhesus coals invite carelessly apply
